<a href="https://colab.research.google.com/github/yhlimath/Spider-action-codes/blob/jules-9207673093865262207-475ed8a0/action_on_magnetic_modules.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1 &nbsp; The spider generator and data structures

## 1.1&nbsp;Data structures

In [ ]:
# the class FormalSumgives the space of strings a module structure over polynomials in n=[2]_q.

class FormalSum:
    """Class to represent formal linear combinations of strings with polynomial coefficients"""
    def __init__(self, terms=None):
        # terms is a dictionary: {tuple(string): Polynomial coefficient}
        self.terms = terms if terms is not None else {}
        self._normalize()

    def _normalize(self):
        """Remove terms with zero coefficients and combine like terms"""
        normalized_terms = {}
        for key, coeff in self.terms.items():
            if isinstance(coeff, (int, float)):
                if coeff != 0:
                    normalized_terms[key] = Polynomial.constant(coeff)
            elif isinstance(coeff, Polynomial):
                if not coeff.is_zero():
                    normalized_terms[key] = coeff
            else:
                # Handle other coefficient types
                normalized_terms[key] = coeff
        self.terms = normalized_terms

    def add_term(self, string, coeff=None):
        """Add a term to the formal sum"""
        if coeff is None:
            coeff = Polynomial.constant(1)
        elif isinstance(coeff, (int, float)):
            coeff = Polynomial.constant(coeff)

        key = tuple(string)
        if key in self.terms:
            self.terms[key] = self.terms[key] + coeff
        else:
            self.terms[key] = coeff
        self._normalize()

    def add_coefficient_string_pairs(self, pairs):
        """Add multiple (coefficient, string) pairs"""
        for coeff, string in pairs:
            self.add_term(string, coeff)

    def __add__(self, other):
        """Add two formal sums"""
        result = FormalSum()
        result.terms = {k: v for k, v in self.terms.items()}
        for key, coeff in other.terms.items():
            if key in result.terms:
                result.terms[key] = result.terms[key] + coeff
            else:
                result.terms[key] = coeff
        result._normalize()
        return result

    def __sub__(self, other):
        """Subtract two formal sums"""
        result = FormalSum()
        result.terms = {k: v for k, v in self.terms.items()}
        for key, coeff in other.terms.items():
            if key in result.terms:
                result.terms[key] = result.terms[key] - coeff
            else:
                result.terms[key] = -coeff
        result._normalize()
        return result

    def __eq__(self, other):
        """Check if two formal sums are equal"""
        return self.terms == other.terms

    def __str__(self):
        if not self.terms:
            return "0"
        parts = []
        for key, coeff in sorted(self.terms.items()):
            string = list(key)
            coeff_str = str(coeff)
            if coeff == Polynomial.constant(1):
                parts.append(f"{string}")
            elif coeff == Polynomial.constant(-1):
                parts.append(f"-{string}")
            else:
                parts.append(f"({coeff})*{string}")
        result = " + ".join(parts).replace("+ -", "- ")
        return result

    def __repr__(self):
        return f"FormalSum({self.terms})"

    def is_zero(self):
        """Check if the formal sum is zero"""
        return len(self.terms) == 0

def apply_e_to_formal_sum(formal_sum, i):
    """Apply operation e_i to a formal sum"""
    result = FormalSum()

    for string_tuple, coeff in formal_sum.terms.items():
        string = list(string_tuple)
        # Convert to the format expected by e function
        input_pairs = [(coeff, string)]
        # Apply e operation
        output_pairs = e(input_pairs, i, verbose=False)
        # Add results to formal sum
        result.add_coefficient_string_pairs(output_pairs)

    return result

def apply_sequence_to_formal_sum(formal_sum, operations):
    """Apply a sequence of e operations to a formal sum"""
    current = formal_sum
    for op in operations:
        current = apply_e_to_formal_sum(current, op)
    return current



def create_coefficient_string_pair(coeff, string):
    """Helper function to create (coefficient, string) pairs"""
    return (coeff, string)

def display_coefficient_strings(S):
    """Helper function to nicely display a list of (coefficient, string) pairs"""
    if not S:
        return "[]"
    parts = []
    for coeff, string in S:
        if coeff == 1:
            parts.append(f"{string}")
        elif coeff == -1:
            parts.append(f"-{string}")
        else:
            parts.append(f"{coeff}*{string}")
    return "[" + ", ".join(parts) + "]"


import random

class Polynomial:
    """Class to represent polynomials in variable n"""
    def __init__(self, coeffs=None):
        # coeffs is a dictionary: {power: coefficient}
        # e.g., n^2 + 3n - 1 would be {2: 1, 1: 3, 0: -1}
        self.coeffs = coeffs if coeffs is not None else {}
        self._normalize()

    def _normalize(self):
        """Remove terms with zero coefficients"""
        self.coeffs = {k: v for k, v in self.coeffs.items() if v != 0}

    @classmethod
    def constant(cls, value):
        """Create a constant polynomial"""
        return cls({0: value}) if value != 0 else cls({})

    @classmethod
    def variable(cls, power=1):
        """Create n^power"""
        return cls({power: 1})

    def __add__(self, other):
        """Add two polynomials"""
        if isinstance(other, (int, float)):
            other = Polynomial.constant(other)
        result = Polynomial()
        result.coeffs = self.coeffs.copy()
        for power, coeff in other.coeffs.items():
            result.coeffs[power] = result.coeffs.get(power, 0) + coeff
        result._normalize()
        return result

    def __radd__(self, other):
        return self.__add__(other)

    def __sub__(self, other):
        """Subtract two polynomials"""
        if isinstance(other, (int, float)):
            other = Polynomial.constant(other)
        result = Polynomial()
        result.coeffs = self.coeffs.copy()
        for power, coeff in other.coeffs.items():
            result.coeffs[power] = result.coeffs.get(power, 0) - coeff
        result._normalize()
        return result

    def __rsub__(self, other):
        return (Polynomial.constant(other) - self) if isinstance(other, (int, float)) else NotImplemented

    def __mul__(self, other):
        """Multiply two polynomials"""
        if isinstance(other, (int, float)):
            other = Polynomial.constant(other)
        result = Polynomial()
        for p1, c1 in self.coeffs.items():
            for p2, c2 in other.coeffs.items():
                power = p1 + p2
                result.coeffs[power] = result.coeffs.get(power, 0) + c1 * c2
        result._normalize()
        return result

    def __rmul__(self, other):
        return self.__mul__(other)

    def __neg__(self):
        """Negate a polynomial"""
        result = Polynomial()
        result.coeffs = {k: -v for k, v in self.coeffs.items()}
        return result

    def __eq__(self, other):
        """Check if two polynomials are equal"""
        if isinstance(other, (int, float)):
            other = Polynomial.constant(other)
        return self.coeffs == other.coeffs

    def __str__(self):
        if not self.coeffs:
            return "0"

        parts = []
        for power in sorted(self.coeffs.keys(), reverse=True):
            coeff = self.coeffs[power]

            if power == 0:
                if coeff > 0 and parts:
                    parts.append(f"+{coeff}")
                else:
                    parts.append(str(coeff))
            elif power == 1:
                if coeff == 1:
                    parts.append("+n" if parts else "n")
                elif coeff == -1:
                    parts.append("-n")
                else:
                    if coeff > 0 and parts:
                        parts.append(f"+{coeff}n")
                    else:
                        parts.append(f"{coeff}n")
            else:
                if coeff == 1:
                    parts.append(f"+n^{power}" if parts else f"n^{power}")
                elif coeff == -1:
                    parts.append(f"-n^{power}")
                else:
                    if coeff > 0 and parts:
                        parts.append(f"+{coeff}n^{power}")
                    else:
                        parts.append(f"{coeff}n^{power}")

        return "".join(parts)

    def __repr__(self):
        return f"Polynomial({self.coeffs})"

    def is_zero(self):
        """Check if polynomial is zero"""
        return len(self.coeffs) == 0

    def evaluate(self, n_value):
        """Evaluate polynomial at a specific value of n"""
        result = 0
        for power, coeff in self.coeffs.items():
            result += coeff * (n_value ** power)
        return result


## 1.2&nbsp;MAIN CODES defining the generators e and f

In [ ]:

def generate_balanced_string(n):
    """Generate a shuffled string of length 3n with equal numbers of 1, 0, and -1."""
    numbers = [1] * n + [0] * n + [-1] * n
    random.shuffle(numbers)
    return numbers

def generate_all_valid_strings(n):
    """Generate all valid strings of length 3n that correspond to walks in the sl_3 Weyl chamber """
    def backtrack(sequence, count_1, count_0, count_neg1, results):
        if len(sequence) == 3 * n:
            results.append(sequence[:])  # Store a copy of the valid sequence
            return
        if count_1 < n:
            backtrack(sequence + [1], count_1 + 1, count_0, count_neg1, results)
        if count_0 < count_1 and count_0 < n:
            backtrack(sequence + [0], count_1, count_0 + 1, count_neg1, results)
        if count_neg1 < count_0 and count_neg1 < n:
            backtrack(sequence + [-1], count_1, count_0, count_neg1 + 1, results)

    results = []
    backtrack([], 0, 0, 0, results)
    return results

def form_triplets_with_positions(sequence):
    """Form triplets from the sequence using the specified rightmost-1-first rule,
    and return the positions of the numbers in the original sequence."""
    sequence = sequence[:]
    indexed_sequence = list(enumerate(sequence))  # Store original positions
    triplets = []

    while indexed_sequence:
        try:
            index_1 = max(i for i, val in indexed_sequence if val == 1)
        except ValueError:
            break  # No more 1s left
        try:
            index_0 = next(i for i, val in indexed_sequence if val == 0 and i > index_1)
        except StopIteration:
            break  # No more 0s left
        try:
            index_neg1 = next(i for i, val in indexed_sequence if val == -1 and i > index_0)
        except StopIteration:
            break  # No more -1s left

        triplet = ((1, index_1), (0, index_0), (-1, index_neg1))
        triplets.append(triplet)

        indexed_sequence = [(i, val) for i, val in indexed_sequence if i not in {index_1, index_0, index_neg1}]

    return triplets

def bend_string(sequence):
    """Perform the bending operation on the given sequence."""
    sequence = sequence[:]
    indexed_sequence = list(enumerate(sequence))  # Store original positions

    try:
        start_1_index = next(i for i, val in indexed_sequence if val == 1)
    except StopIteration:
        return sequence  # No 1 found, return unchanged

    triplets = form_triplets_with_positions(sequence)
    for triplet in triplets:
        if triplet[0][0] == 1 and triplet[0][1] == start_1_index:
            index_1, index_0, index_neg1 = triplet[0][1], triplet[1][1], triplet[2][1]
            break
    else:
        return sequence  # No valid triplet found, return unchanged

    new_sequence = sequence[:]
    new_sequence[index_0] = 1
    new_sequence[index_neg1] = 0
    new_sequence.pop(index_1)
    new_sequence.append(-1)

    return new_sequence

def bending_power(s, p):
    """Applies the bending operation to string s for p times."""
    s_bent_p_times = s[:]  # Create a copy of s
    for _ in range(p):
        s_bent_p_times = bend_string(s_bent_p_times)
    return s_bent_p_times

def inverse_bending_power(s, p):
    """Applies the inverse bending operation to string s for p times."""
    return bending_power(s, len(s) - p)

def generate_triplet_at_position(s, p):
    if p > len(s):
        # If p is larger than the length of s, append the triplet at the end
        return s + [1, 0, -1]
    else:
        # Otherwise, insert the triplet at position p
        return s[:p-1] + [1, 0, -1] + s[p-1:]

def find_last_triplet(s):
    """Finds the last triplet (1, 0, -1) in the string s, containing the last 1."""
    try:
        last_1_index = len(s) - 1 - s[::-1].index(1)  # Find index of last 1
        last_0_index = next(i for i in range(last_1_index + 1, len(s)) if s[i] == 0)
        last_neg1_index = next(i for i in range(last_0_index + 1, len(s)) if s[i] == -1)
        return last_1_index, last_0_index, last_neg1_index
    except (ValueError, StopIteration):
        return None

def string_decomposition(s):
    s_decomposed = s[:]  # Create a copy of s to work with
    operations = []  # Initialize an empty list to store operations

    while len(s_decomposed) > 6:
        last_triplet_indices = find_last_triplet(s_decomposed)
        if last_triplet_indices is None:
            break  # No more triplets found, terminate

        last_1_index, last_0_index, last_neg1_index = last_triplet_indices

        if last_0_index == last_1_index + 1 and last_neg1_index == last_0_index + 1:
            # If last triplet occupies consecutive sites, delete it
            s_decomposed = s_decomposed[:last_1_index] + s_decomposed[last_neg1_index + 1:]
            operations.append(('t', last_1_index + 1))  # Record operation

        else:
            # Handle the case where the last triplet is not consecutive
            if all(x == -1 for x in s_decomposed[last_1_index + 1:last_0_index]):
                # If there is a string of -1s between 1 and 0
                start_neg1_string = last_1_index + 1
                end_neg1_string = last_0_index - 1

                # Perform cyclic permutation: 1, -1, ... , -1 to -1, ... , -1, 1
                s_decomposed[last_1_index], s_decomposed[end_neg1_string] = s_decomposed[end_neg1_string], s_decomposed[last_1_index]

                # Update operations with (e, position of -1 - 1) in increasing order of -1 positions
                for i in range(start_neg1_string, end_neg1_string + 1):
                    operations.append(('e', i))

                #Handle 0s between 0 and -1 simultaneously:
            if all(x == 0 for x in s_decomposed[last_0_index + 1:last_neg1_index]):
                    start_0_string = last_0_index + 1
                    end_0_string = last_neg1_index - 1

                    # Perform cyclic permutation for 0s:
                    s_decomposed[last_neg1_index], s_decomposed[start_0_string] = s_decomposed[start_0_string] ,s_decomposed[last_neg1_index]

                    # Record operations for 0s in decreasing order of positions:
                    for i in range(end_0_string, start_0_string -1, -1): #start_0_string included
                        operations.append(('e', i+1))

            else:
                # Handle other non-consecutive cases
                pass  # Placeholder for now

    return s_decomposed, operations

def e(S, i, verbose=False):
    """
    Apply operation e_i to a list of (coefficient, string) pairs.

    Args:
        S: List of (coefficient, string) pairs
        i: Position for the e operation
        verbose: Whether to print detailed output

    Returns:
        List of (coefficient, string) pairs
    """
    S_out = []

    for coeff, s in S:
        if verbose:
            print(f"Computing e({coeff} * {s}, {i})")

        bent_s = bending_power(s, i-1)
        if verbose:
            print(f"Bent string B^{i-1}({s}): {bent_s}")

        s_decomposed, operations = string_decomposition(bent_s)
        if verbose:
            print(f"Decomposing B^{i-1}({s}) gives s_decomposed= {s_decomposed} and operations={operations}")

        if s_decomposed in [[1, 0, 1, 0, -1, -1], [1, 0, 1, -1, 0, -1], [1, 0, -1, 1, 0, -1]]:
            # Return original string with coefficient multiplied by n
            if isinstance(coeff, (int, float)):
                new_coeff = Polynomial.constant(coeff) * Polynomial.variable()
            elif isinstance(coeff, Polynomial):
                new_coeff = coeff * Polynomial.variable()
            else:
                # Fallback for other types
                new_coeff = coeff

            S_out.append((new_coeff, s))
            if verbose:
                print(f"e({coeff} * {s}, {i}) -> {new_coeff} * {s} (ends at original string, coefficient multiplied by n)")

        elif s_decomposed == [1, 1, 0, -1, 0, -1]:
            # Find the triplet containing the 1 on the second entry
            triplets = form_triplets_with_positions(bent_s)
            target_triplet = None
            for triplet in triplets:
                if triplet[0][0] == 1 and triplet[0][1] == bent_s.index(1, 1):  # Find the second 1
                    target_triplet = triplet
                    break

            # Create s_jescon by swapping 1 and 0 in the target triplet
            if target_triplet:
                s_jescon = bent_s[:]
                index_1, index_0, _ = target_triplet[0][1], target_triplet[1][1], target_triplet[2][1]
                s_jescon[index_1], s_jescon[index_0] = s_jescon[index_0], s_jescon[index_1]
                result = inverse_bending_power(s_jescon, i - 1)
                S_out.append((coeff, result))
                if verbose:
                    print(f"e({coeff} * {s}, {i}) -> {coeff} * {result} (Jesper's conjecture)")

        elif s_decomposed == [1,1,0,0,-1,-1]:
            new_strings = [[1,0,-1,1,0,-1],[1,0,1,0,-1,-1]]
            if verbose:
                print(f"Action of e_1 creates a square here, s_decomposed splits into new strings")

            # Apply operations in reverse order to both new strings
            for op_type, position in reversed(operations):
                if op_type == 't':
                    temp_new_strings = []
                    for new_s in new_strings:
                        temp_new_strings.append(generate_triplet_at_position(new_s, position))
                    new_strings = temp_new_strings
                elif op_type == 'e':
                    # Convert to (coefficient, string) format for recursive call
                    temp_input = [(1, new_s) for new_s in new_strings]  # Coefficient 1 for intermediate
                    temp_result = e(temp_input, position, verbose=False)
                    # Extract just the strings (coefficients should still be 1)
                    new_strings = [string for _, string in temp_result]

            # Apply inverse bending and add to output with original coefficient
            final_results = [inverse_bending_power(new_s, i - 1) for new_s in new_strings]
            for result in final_results:
                S_out.append((coeff, result))
            if verbose:
                print(f"e({coeff} * {s}, {i}) -> {[f'{coeff} * {res}' for res in final_results]} (splitting case)")

    return S_out



def f(S, i, verbose=False):
    """
    Apply operation f_i to a list of (coefficient, string) pairs.
    f_i(S) = e_i e_{i+1} e_i(S) - e_i(S)

    Args:
        S: List of (coefficient, string) pairs
        i: Position for the f operation
        verbose: Whether to print detailed output

    Returns:
        List of (coefficient, string) pairs representing the formal sum
    """
    if verbose:
        print(f"Computing f({display_coefficient_strings(S)}, {i})")
        print(f"f_{i} = e_{i} e_{{{i+1}}} e_{i} - e_{i}")

    # Convert input to FormalSum for easier manipulation
    input_formal_sum = FormalSum()
    input_formal_sum.add_coefficient_string_pairs(S)

    if verbose:
        print(f"Input as formal sum: {input_formal_sum}")

    # Compute e_i e_{i+1} e_i(S)
    term1 = apply_sequence_to_formal_sum(input_formal_sum, [i, i+1, i])
    if verbose:
        print(f"e_{i} e_{{{i+1}}} e_{i}(S) = {term1}")

    # Compute e_i(S)
    term2 = apply_sequence_to_formal_sum(input_formal_sum, [i])
    if verbose:
        print(f"e_{i}(S) = {term2}")

    # Compute the difference: e_i e_{i+1} e_i(S) - e_i(S)
    result_formal_sum = term1 - term2
    if verbose:
        print(f"f_{i}(S) = e_{i} e_{{{i+1}}} e_{i}(S) - e_{i}(S) = {result_formal_sum}")

    # Convert back to list of (coefficient, string) pairs
    result_pairs = []
    for string_tuple, coeff in result_formal_sum.terms.items():
        result_pairs.append((coeff, list(string_tuple)))

    return result_pairs









# 2&nbsp;Action on magnetic modules

##2.1&nbsp; Definition of magnetic modules, and a mapping to the vacuum module

In [ ]:

def generate_constrained_strings(m, x, y):
    """Generate all valid strings of type (m,x,y)"""
    if (m + 2*x + y) % 3 != 0:
        return []

    count_1s = (m + 2*x + y) // 3
    count_0s = (m - x + y) // 3
    count_minus1s = (m - x - 2*y) // 3

    if count_1s < 0 or count_0s < 0 or count_minus1s < 0:
        return []

    if count_1s + count_0s + count_minus1s != m:
        return []

    def backtrack(sequence, remaining_1s, remaining_0s, remaining_minus1s,
                  current_1s, current_0s, current_minus1s, results):
        if len(sequence) == m:
            if (remaining_1s == 0 and remaining_0s == 0 and remaining_minus1s == 0):
                results.append(sequence[:])
            return

        if remaining_1s > 0:
            new_1s = current_1s + 1
            if new_1s >= current_0s >= current_minus1s:
                backtrack(sequence + [1], remaining_1s - 1, remaining_0s, remaining_minus1s,
                         new_1s, current_0s, current_minus1s, results)

        if remaining_0s > 0:
            new_0s = current_0s + 1
            if current_1s >= new_0s >= current_minus1s:
                backtrack(sequence + [0], remaining_1s, remaining_0s - 1, remaining_minus1s,
                         current_1s, new_0s, current_minus1s, results)

        if remaining_minus1s > 0:
            new_minus1s = current_minus1s + 1
            if current_1s >= current_0s >= new_minus1s:
                backtrack(sequence + [-1], remaining_1s, remaining_0s, remaining_minus1s - 1,
                         current_1s, current_0s, new_minus1s, results)

    results = []
    backtrack([], count_1s, count_0s, count_minus1s, 0, 0, 0, results)
    return results

def map_constrained_to_balanced(s, x, y):
    """Map constrained string to balanced string"""
    s_prime = s.copy()
    for _ in range(x):
        s_prime.extend([0, -1])
    for _ in range(y):
        s_prime.append(-1)
    return s_prime

def is_valid_constrained_string(string, m, x, y):
    """Check if string satisfies (m,x,y) constraints"""
    if len(string) != m:
        return False

    count_1, count_0, count_neg1 = 0, 0, 0
    for element in string:
        if element == 1:
            count_1 += 1
        elif element == 0:
            count_0 += 1
        elif element == -1:
            count_neg1 += 1
        else:
            return False

        if not (count_1 >= count_0 >= count_neg1):
            return False

    final_x = count_1 - count_0
    final_y = count_0 - count_neg1

    return final_x == x and final_y == y

In [ ]:
generate_constrained_strings(4,1,0)

[[1, 1, 0, -1], [1, 0, 1, -1], [1, 0, -1, 1]]

##2.2&nbsp; e generator on magnetic modules

In [ ]:
def ed(S, i, m, x, y, verbose=False):
    """Extended e function for constrained strings"""
    if not S:
        return []

    # Step 1: Map to balanced strings
    balanced_pairs = []
    for coeff, constrained_string in S:
        balanced_string = map_constrained_to_balanced(constrained_string, x, y)
        balanced_pairs.append((coeff, balanced_string))

    # Step 2: Apply e function
    balanced_results = e(balanced_pairs, i, verbose)

    # Step 3: Truncate and filter
    final_results = []
    for coeff, balanced_result in balanced_results:
        if len(balanced_result) >= m:
            truncated = balanced_result[:m]
            if is_valid_constrained_string(truncated, m, x, y):
                final_results.append((coeff, truncated))

    return final_results


# 3&nbsp; Testing algebraic relations on magnetic modules